# 03. Q-Learning: Aprendizaje sin Modelo

**Nivel:** 🟡 Intermedio  
**Tiempo estimado:** 90 minutos  
**Prerequisitos:** [02. MDP y Ecuaciones de Bellman](02-mdp-bellman.ipynb)

## 🎯 Objetivos de Aprendizaje
Al finalizar este notebook, podrás:
- Comprender la diferencia entre métodos model-based y model-free
- Implementar el algoritmo Q-Learning desde cero
- Entender el trade-off entre exploración y explotación (ε-greedy)
- Analizar la convergencia de Q-Learning
- Aplicar Q-Learning a problemas clásicos (FrozenLake, Taxi)
- Visualizar el proceso de aprendizaje de la tabla Q

## 📚 Motivación

### El Problema del Modelo Desconocido

En el notebook anterior, usamos **programación dinámica** (Value Iteration, Policy Iteration) para encontrar políticas óptimas. Sin embargo, estos métodos tienen una limitación fundamental:

**Requieren conocer la dinámica del ambiente**: las funciones $P(s'|s,a)$ y $R(s,a,s')$.

En muchos problemas reales, **no conocemos** estas funciones:
- Un robot navegando en un ambiente nuevo
- Un agente jugando un videojuego sin acceso al código del juego
- Un sistema de trading en mercados financieros
- Un robot aprendiendo a caminar

### ¿Cómo aprender sin conocer el modelo?

La solución: **Aprendizaje por experiencia (sampling)**.

En lugar de calcular:
$$V(s) = \max_a \sum_{s'} P(s'|s,a)[R(s,a,s') + \gamma V(s')]$$

Estimamos usando muestras:
$$Q(s,a) \leftarrow Q(s,a) + \alpha [r + \gamma \max_{a'} Q(s',a') - Q(s,a)]$$

Esto es **Q-Learning**: aprende la función Q óptima directamente de la experiencia.

### Pregunta Guía
**¿Cómo puede un agente aprender la política óptima sin conocer las reglas del ambiente, usando solo las recompensas que recibe?**

Q-Learning responde exactamente esta pregunta y es uno de los algoritmos más importantes en RL.

In [ ]:
# Importar librerías
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import gymnasium as gym
from typing import Tuple, List, Dict
from collections import defaultdict
import pandas as pd
from tqdm import tqdm

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline
np.random.seed(42)

print("✅ Librerías importadas correctamente")

## 🎨 Intuición Visual

### Model-Based vs Model-Free

Veamos la diferencia fundamental entre ambos enfoques:

In [ ]:
from IPython.display import display, HTML

comparison = """
<table style='width:100%; border-collapse: collapse; font-size: 14px;'>
  <tr style='background-color: #f0f0f0;'>
    <th style='border: 1px solid black; padding: 10px;'>Aspecto</th>
    <th style='border: 1px solid black; padding: 10px;'>Model-Based (DP)</th>
    <th style='border: 1px solid black; padding: 10px;'>Model-Free (Q-Learning)</th>
  </tr>
  <tr>
    <td style='border: 1px solid black; padding: 10px;'><b>Conocimiento</b></td>
    <td style='border: 1px solid black; padding: 10px;'>Requiere P(s'|s,a) y R(s,a,s')</td>
    <td style='border: 1px solid black; padding: 10px;'>Solo necesita interactuar</td>
  </tr>
  <tr>
    <td style='border: 1px solid black; padding: 10px;'><b>Método</b></td>
    <td style='border: 1px solid black; padding: 10px;'>Cálculo exacto (sumas)</td>
    <td style='border: 1px solid black; padding: 10px;'>Estimación por muestreo</td>
  </tr>
  <tr>
    <td style='border: 1px solid black; padding: 10px;'><b>Eficiencia</b></td>
    <td style='border: 1px solid black; padding: 10px;'>Converge rápido si conoce el modelo</td>
    <td style='border: 1px solid black; padding: 10px;'>Más lento, pero más general</td>
  </tr>
  <tr>
    <td style='border: 1px solid black; padding: 10px;'><b>Aplicabilidad</b></td>
    <td style='border: 1px solid black; padding: 10px;'>Ambientes pequeños conocidos</td>
    <td style='border: 1px solid black; padding: 10px;'>Cualquier ambiente</td>
  </tr>
  <tr>
    <td style='border: 1px solid black; padding: 10px;'><b>Ejemplo</b></td>
    <td style='border: 1px solid black; padding: 10px;'>GridWorld con reglas dadas</td>
    <td style='border: 1px solid black; padding: 10px;'>Atari, Robótica, Juegos</td>
  </tr>
</table>
"""

display(HTML(comparison))

print("\n💡 Insight Clave:")
print("  Q-Learning sacrifica eficiencia por generalidad.")
print("  No necesita el modelo → aplicable a problemas reales complejos.")

### Visualización de la Tabla Q

La **tabla Q** almacena $Q(s,a)$ para cada par estado-acción:

In [ ]:
def visualize_q_table_concept():
    """
    Visualiza el concepto de una tabla Q con un ejemplo simple.
    """
    # Crear tabla Q de ejemplo (4 estados, 4 acciones)
    q_table = np.array([
        [0.1, 0.5, 0.3, 0.2],  # Estado 0
        [0.4, 0.8, 0.6, 0.3],  # Estado 1
        [0.7, 0.9, 0.5, 0.4],  # Estado 2
        [0.2, 0.3, 0.1, 1.0],  # Estado 3 (cerca del objetivo)
    ])
    
    actions = ['↑', '→', '↓', '←']
    states = ['S0', 'S1', 'S2', 'S3']
    
    fig = go.Figure(data=go.Heatmap(
        z=q_table,
        x=actions,
        y=states,
        colorscale='Viridis',
        text=np.round(q_table, 2),
        texttemplate='%{text}',
        textfont={"size": 14},
        colorbar=dict(title="Q-Value")
    ))
    
    fig.update_layout(
        title='Tabla Q: Q(s,a) para cada Estado-Acción',
        xaxis_title='Acción',
        yaxis_title='Estado',
        width=600,
        height=400
    )
    
    return fig

fig = visualize_q_table_concept()
fig.show()

print("\n📊 Interpretación:")
print("  - Cada celda representa Q(s,a): valor de tomar acción a en estado s")
print("  - Valores más altos (amarillo) = mejores acciones")
print("  - La política óptima elige la acción con máximo Q en cada estado")
print("  - Q-Learning aprende estos valores mediante experiencia")

### Exploración vs Explotación

Un dilema fundamental en RL:

In [ ]:
def visualize_epsilon_greedy():
    """
    Visualiza cómo funciona la estrategia ε-greedy.
    """
    epsilons = [0.1, 0.3, 0.5, 0.7, 0.9]
    exploit = [1 - e for e in epsilons]
    explore = epsilons
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        name='Explotación (mejor acción conocida)',
        x=[f'ε={e}' for e in epsilons],
        y=exploit,
        marker_color='lightgreen'
    ))
    
    fig.add_trace(go.Bar(
        name='Exploración (acción aleatoria)',
        x=[f'ε={e}' for e in epsilons],
        y=explore,
        marker_color='lightcoral'
    ))
    
    fig.update_layout(
        title='Estrategia ε-Greedy: Balance Exploración-Explotación',
        xaxis_title='Valor de ε',
        yaxis_title='Probabilidad',
        barmode='stack',
        yaxis=dict(range=[0, 1]),
        template='plotly_white'
    )
    
    return fig

fig = visualize_epsilon_greedy()
fig.show()

print("\n🎯 ε-Greedy Strategy:")
print("  - Con probabilidad ε: EXPLORAR (acción aleatoria)")
print("  - Con probabilidad 1-ε: EXPLOTAR (mejor acción conocida)")
print("  \n  Típicamente: ε alto al inicio → ε bajo al final")
print("  Esto permite descubrir el ambiente primero, luego optimizar.")

## 📐 Fundamentos Matemáticos

### Temporal Difference Learning

Q-Learning pertenece a la familia de métodos **Temporal Difference (TD)**.

#### Error TD

$$
\begin{align}
\delta_t &= r_{t+1} + \gamma \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t) \tag{1} \\
\text{donde: } & \\
\delta_t &: \text{error de diferencia temporal} \\
r_{t+1} &: \text{recompensa observada} \\
\gamma \max_{a'} Q(s_{t+1}, a') &: \text{mejor valor futuro estimado} \\
Q(s_t, a_t) &: \text{estimación actual}
\end{align}
$$

**Interpretación**: 
- Si $\delta_t > 0$: La acción fue mejor de lo esperado → aumentar Q
- Si $\delta_t < 0$: La acción fue peor de lo esperado → disminuir Q

### Algoritmo Q-Learning

#### Regla de Actualización

$$
\begin{align}
Q(s_t, a_t) &\leftarrow Q(s_t, a_t) + \alpha \delta_t \tag{2} \\
            &= Q(s_t, a_t) + \alpha [r_{t+1} + \gamma \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t)] \tag{3} \\
            &= (1 - \alpha) Q(s_t, a_t) + \alpha [r_{t+1} + \gamma \max_{a'} Q(s_{t+1}, a')] \tag{4} \\
\text{donde: } & \\
\alpha &\in (0, 1]: \text{tasa de aprendizaje (learning rate)} \\
\gamma &\in [0, 1]: \text{factor de descuento}
\end{align}
$$

**La ecuación (4) muestra que Q es un promedio ponderado entre:**
- Estimación antigua: $Q(s_t, a_t)$
- Nueva estimación (target): $r_{t+1} + \gamma \max_{a'} Q(s_{t+1}, a')$

#### Pseudocódigo

```
Inicializar Q(s,a) arbitrariamente
Para cada episodio:
    Inicializar s
    Para cada paso del episodio:
        Elegir a desde s usando política ε-greedy derivada de Q
        Tomar acción a, observar r, s'
        Q(s,a) ← Q(s,a) + α[r + γ max_a' Q(s',a') - Q(s,a)]
        s ← s'
    Hasta que s sea terminal
```

### Convergencia

**Teorema (Watkins & Dayan, 1992)**: Q-Learning converge a $Q^*$ con probabilidad 1 si:

1. Todos los pares (s,a) son visitados infinitamente
2. $\sum_{t=1}^{\infty} \alpha_t = \infty$ (suma de alphas diverge)
3. $\sum_{t=1}^{\infty} \alpha_t^2 < \infty$ (suma de alphas al cuadrado converge)

**Condiciones típicas satisfacen esto**:
- $\alpha$ constante pequeño (e.g., 0.1)
- $\alpha_t = 1/t$ (decreciente)
- Exploración ε-greedy con ε > 0

### Ejemplo Numérico

Supongamos:
- Estado actual: $s_0$
- Acción: $a_0$ (derecha)
- Q actual: $Q(s_0, a_0) = 0.5$
- Recompensa: $r = -0.1$
- Siguiente estado: $s_1$
- Mejor Q en $s_1$: $\max_{a'} Q(s_1, a') = 0.8$
- Parámetros: $\alpha = 0.1$, $\gamma = 0.9$

**Cálculo del error TD**:
$$
\begin{align}
\delta &= r + \gamma \max_{a'} Q(s_1, a') - Q(s_0, a_0) \\
       &= -0.1 + 0.9 \times 0.8 - 0.5 \\
       &= -0.1 + 0.72 - 0.5 \\
       &= 0.12
\end{align}
$$

**Actualización de Q**:
$$
\begin{align}
Q(s_0, a_0) &\leftarrow Q(s_0, a_0) + \alpha \delta \\
            &= 0.5 + 0.1 \times 0.12 \\
            &= 0.5 + 0.012 \\
            &= 0.512
\end{align}
$$

El Q-value aumentó ligeramente porque el camino fue mejor de lo esperado ($\delta > 0$).

> 💡 **Insight**: Q-Learning usa una sola transición (sample) para actualizar, en lugar de sumar sobre todas las transiciones posibles como en DP.

## 💻 Implementación Desde Cero

Implementaremos Q-Learning completo y lo probaremos en diferentes ambientes.

In [ ]:
class QLearningAgent:
    """
    Agente que implementa Q-Learning tabular.
    
    Mantiene una tabla Q(s,a) y la actualiza usando la regla de Q-Learning.
    Usa estrategia ε-greedy para balancear exploración y explotación.
    
    Parameters:
    -----------
    n_states : int
        Número de estados en el ambiente
    n_actions : int
        Número de acciones posibles
    learning_rate : float
        Tasa de aprendizaje α ∈ (0, 1]
    discount_factor : float
        Factor de descuento γ ∈ [0, 1]
    epsilon : float
        Probabilidad de exploración ε ∈ [0, 1]
    epsilon_decay : float
        Factor de decaimiento de ε por episodio
    epsilon_min : float
        Valor mínimo de ε
    """
    
    def __init__(self,
                 n_states: int,
                 n_actions: int,
                 learning_rate: float = 0.1,
                 discount_factor: float = 0.99,
                 epsilon: float = 1.0,
                 epsilon_decay: float = 0.995,
                 epsilon_min: float = 0.01):
        
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = learning_rate
        self.gamma = discount_factor
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        
        # Inicializar tabla Q a ceros
        self.q_table = np.zeros((n_states, n_actions))
        
        # Para tracking
        self.training_history = {
            'episode': [],
            'total_reward': [],
            'epsilon': [],
            'steps': []
        }
    
    def select_action(self, state: int, training: bool = True) -> int:
        """
        Selecciona una acción usando estrategia ε-greedy.
        
        Parameters:
        -----------
        state : int
            Estado actual
        training : bool
            Si True, usa ε-greedy. Si False, siempre greedy (para evaluación)
        
        Returns:
        --------
        action : int
            Acción seleccionada
        """
        # Exploración: acción aleatoria
        if training and np.random.random() < self.epsilon:
            return np.random.randint(0, self.n_actions)
        
        # Explotación: mejor acción según Q-table
        return np.argmax(self.q_table[state])
    
    def update(self, state: int, action: int, reward: float, next_state: int, done: bool):
        """
        Actualiza la tabla Q usando la regla de Q-Learning.
        
        Q(s,a) ← Q(s,a) + α[r + γ max_a' Q(s',a') - Q(s,a)]
        
        Parameters:
        -----------
        state : int
            Estado actual
        action : int
            Acción tomada
        reward : float
            Recompensa recibida
        next_state : int
            Siguiente estado
        done : bool
            Si el episodio terminó
        """
        # Q-value actual
        current_q = self.q_table[state, action]
        
        # Target: r + γ max_a' Q(s',a')
        # Si el episodio terminó, no hay valor futuro
        if done:
            target = reward
        else:
            target = reward + self.gamma * np.max(self.q_table[next_state])
        
        # Error TD
        td_error = target - current_q
        
        # Actualización de Q
        self.q_table[state, action] = current_q + self.alpha * td_error
    
    def decay_epsilon(self):
        """
        Decae epsilon después de cada episodio.
        """
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
    
    def train(self, env, n_episodes: int = 1000, max_steps: int = 100, verbose: bool = True):
        """
        Entrena el agente en el ambiente.
        
        Parameters:
        -----------
        env : gym.Env
            Ambiente de Gymnasium
        n_episodes : int
            Número de episodios de entrenamiento
        max_steps : int
            Máximo de pasos por episodio
        verbose : bool
            Si True, imprime progreso
        """
        for episode in range(n_episodes):
            state, _ = env.reset()
            total_reward = 0
            
            for step in range(max_steps):
                # Seleccionar acción
                action = self.select_action(state, training=True)
                
                # Ejecutar acción
                next_state, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated
                
                # Actualizar Q-table
                self.update(state, action, reward, next_state, done)
                
                total_reward += reward
                state = next_state
                
                if done:
                    break
            
            # Decaer epsilon
            self.decay_epsilon()
            
            # Registrar métricas
            self.training_history['episode'].append(episode)
            self.training_history['total_reward'].append(total_reward)
            self.training_history['epsilon'].append(self.epsilon)
            self.training_history['steps'].append(step + 1)
            
            # Imprimir progreso
            if verbose and (episode + 1) % 100 == 0:
                avg_reward = np.mean(self.training_history['total_reward'][-100:])
                print(f"Episodio {episode + 1}/{n_episodes} | "
                      f"Recompensa promedio (últimos 100): {avg_reward:.2f} | "
                      f"ε: {self.epsilon:.4f}")
        
        if verbose:
            print("\n✅ Entrenamiento completado!")

print("✅ Clase QLearningAgent implementada")

### Entrenando en FrozenLake

In [ ]:
# Crear ambiente FrozenLake
env = gym.make('FrozenLake-v1', map_name="4x4", is_slippery=False)

print("🧊 FrozenLake-v1 Environment")
print(f"  Estados: {env.observation_space.n}")
print(f"  Acciones: {env.action_space.n}")
print("  Objetivo: Llegar al objetivo (G) sin caer en agujeros (H)\n")

# Crear agente
agent = QLearningAgent(
    n_states=env.observation_space.n,
    n_actions=env.action_space.n,
    learning_rate=0.1,
    discount_factor=0.99,
    epsilon=1.0,
    epsilon_decay=0.995,
    epsilon_min=0.01
)

print("🤖 Agente Q-Learning creado")
print(f"  Learning rate α: {agent.alpha}")
print(f"  Discount factor γ: {agent.gamma}")
print(f"  Epsilon inicial: {agent.epsilon}\n")

# Entrenar
print("🎯 Iniciando entrenamiento...\n")
agent.train(env, n_episodes=2000, max_steps=100, verbose=True)

env.close()

### Visualizando el Aprendizaje

In [ ]:
def plot_training_history(agent: QLearningAgent):
    """
    Visualiza el progreso del entrenamiento.
    """
    history = agent.training_history
    
    # Calcular promedios móviles
    window = 50
    rewards_smooth = pd.Series(history['total_reward']).rolling(window=window, min_periods=1).mean()
    
    # Crear subplots
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('Recompensa por Episodio', 'Epsilon (Exploración) a lo Largo del Tiempo'),
        vertical_spacing=0.12
    )
    
    # Plot 1: Recompensas
    fig.add_trace(
        go.Scatter(
            x=history['episode'],
            y=history['total_reward'],
            mode='lines',
            name='Recompensa',
            line=dict(color='lightblue', width=1),
            opacity=0.5
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=history['episode'],
            y=rewards_smooth,
            mode='lines',
            name=f'Media móvil ({window} eps)',
            line=dict(color='darkblue', width=2)
        ),
        row=1, col=1
    )
    
    # Plot 2: Epsilon
    fig.add_trace(
        go.Scatter(
            x=history['episode'],
            y=history['epsilon'],
            mode='lines',
            name='Epsilon',
            line=dict(color='red', width=2)
        ),
        row=2, col=1
    )
    
    fig.update_xaxes(title_text="Episodio", row=2, col=1)
    fig.update_yaxes(title_text="Recompensa", row=1, col=1)
    fig.update_yaxes(title_text="Epsilon", row=2, col=1)
    
    fig.update_layout(
        height=700,
        showlegend=True,
        template='plotly_white'
    )
    
    return fig

fig = plot_training_history(agent)
fig.show()

print("\n📊 Análisis del entrenamiento:")
print(f"  - Recompensa promedio final (últimos 100 eps): {np.mean(agent.training_history['total_reward'][-100:]):.2f}")
print(f"  - Epsilon final: {agent.epsilon:.4f}")
print(f"  - El agente explora menos con el tiempo (epsilon decrece)")
print(f"  - Las recompensas mejoran conforme aprende")

### Visualizando la Tabla Q Aprendida

In [ ]:
def visualize_learned_q_table(agent: QLearningAgent, env_size: int = 4):
    """
    Visualiza la tabla Q aprendida y la política resultante.
    """
    action_names = ['←', '↓', '→', '↑']
    
    # Extraer política (mejor acción en cada estado)
    policy = np.argmax(agent.q_table, axis=1)
    policy_grid = policy.reshape(env_size, env_size)
    
    # Crear visualización de valores máximos
    max_q_values = np.max(agent.q_table, axis=1).reshape(env_size, env_size)
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Valores Q Máximos por Estado', 'Política Aprendida'),
        specs=[[{'type': 'heatmap'}, {'type': 'heatmap'}]]
    )
    
    # Plot 1: Valores Q
    fig.add_trace(
        go.Heatmap(
            z=max_q_values,
            colorscale='Viridis',
            text=np.round(max_q_values, 2),
            texttemplate='%{text}',
            textfont={"size": 12},
            showscale=True,
            colorbar=dict(x=0.45)
        ),
        row=1, col=1
    )
    
    # Plot 2: Política (flechas)
    policy_symbols = np.array([[action_names[a] for a in row] for row in policy_grid])
    
    fig.add_trace(
        go.Heatmap(
            z=np.zeros_like(max_q_values),
            text=policy_symbols,
            texttemplate='%{text}',
            textfont={"size": 20},
            showscale=False,
            colorscale=[[0, 'white'], [1, 'white']]
        ),
        row=1, col=2
    )
    
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    
    fig.update_layout(
        height=400,
        width=900,
        template='plotly_white'
    )
    
    return fig

fig = visualize_learned_q_table(agent, env_size=4)
fig.show()

print("\n💡 Interpretación:")
print("  - Valores Q más altos (amarillo) indican estados más valiosos")
print("  - Las flechas muestran la mejor acción en cada estado")
print("  - La política apunta hacia el objetivo (esquina inferior derecha)")

### Evaluando el Agente Entrenado

In [ ]:
def evaluate_agent(agent: QLearningAgent, env, n_episodes: int = 100) -> Dict:
    """
    Evalúa el agente entrenado sin exploración.
    """
    total_rewards = []
    success_count = 0
    
    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0
        done = False
        steps = 0
        
        while not done and steps < 100:
            # Siempre greedy (sin exploración)
            action = agent.select_action(state, training=False)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            total_reward += reward
            state = next_state
            steps += 1
        
        total_rewards.append(total_reward)
        if total_reward > 0:  # En FrozenLake, reward=1 significa éxito
            success_count += 1
    
    return {
        'mean_reward': np.mean(total_rewards),
        'std_reward': np.std(total_rewards),
        'success_rate': success_count / n_episodes
    }

# Evaluar
env_eval = gym.make('FrozenLake-v1', map_name="4x4", is_slippery=False)
results = evaluate_agent(agent, env_eval, n_episodes=100)
env_eval.close()

print("🎯 Evaluación del agente entrenado (100 episodios):\n")
print(f"  Recompensa promedio: {results['mean_reward']:.3f} ± {results['std_reward']:.3f}")
print(f"  Tasa de éxito: {results['success_rate']*100:.1f}%")
print("\n✅ El agente ha aprendido una política efectiva!")

## 🔧 Versión con Framework

Probemos Q-Learning en un ambiente más complejo: **Taxi-v3**

In [ ]:
# Crear ambiente Taxi
env_taxi = gym.make('Taxi-v3')

print("🚕 Taxi-v3 Environment\n")
print("Descripción: Un taxi debe recoger pasajeros y dejarlos en su destino.")
print(f"  Estados: {env_taxi.observation_space.n} (500 estados discretos)")
print(f"  Acciones: {env_taxi.action_space.n} (sur, norte, este, oeste, recoger, dejar)")
print("  Recompensas: +20 por entregar, -1 por paso, -10 por acción ilegal\n")

# Crear y entrenar agente
agent_taxi = QLearningAgent(
    n_states=env_taxi.observation_space.n,
    n_actions=env_taxi.action_space.n,
    learning_rate=0.1,
    discount_factor=0.99,
    epsilon=1.0,
    epsilon_decay=0.9995,
    epsilon_min=0.01
)

print("🎯 Entrenando agente en Taxi...\n")
agent_taxi.train(env_taxi, n_episodes=5000, max_steps=200, verbose=True)

# Evaluar
results_taxi = evaluate_agent(agent_taxi, env_taxi, n_episodes=100)

print("\n🎯 Evaluación en Taxi (100 episodios):\n")
print(f"  Recompensa promedio: {results_taxi['mean_reward']:.2f} ± {results_taxi['std_reward']:.2f}")
print(f"  Tasa de éxito: {results_taxi['success_rate']*100:.1f}%")

env_taxi.close()

## 🎯 Ejercicios

### 🟢 Ejercicio 1: Experimentar con Hiperparámetros

Modifica α, γ y ε, y observa cómo afectan el aprendizaje.

In [ ]:
def ejercicio_1_hyperparameters():
    """
    Objetivo: Comprender el impacto de los hiperparámetros.
    
    Instrucciones:
    1. Entrena 3 agentes con diferentes learning rates: 0.01, 0.1, 0.5
    2. Usa FrozenLake con 1000 episodios
    3. Compara las recompensas finales promedio
    4. Retorna un diccionario con los resultados
    """
    # TODO: Tu código aquí
    pass

def test_ejercicio_1():
    resultado = ejercicio_1_hyperparameters()
    assert resultado is not None, "❌ Debes retornar un diccionario"
    assert isinstance(resultado, dict), "❌ El resultado debe ser un diccionario"
    print("✅ ¡Correcto! Has explorado el espacio de hiperparámetros.")
    return True

# test_ejercicio_1()  # Descomenta para probar

### 🟡 Ejercicio 2: Implementar SARSA

SARSA es similar a Q-Learning pero es on-policy (usa la acción realmente tomada).

In [ ]:
class SARSAAgent(QLearningAgent):
    """
    Implementa SARSA (State-Action-Reward-State-Action).
    
    Diferencia con Q-Learning:
    - Q-Learning: Q(s,a) ← Q(s,a) + α[r + γ max_a' Q(s',a') - Q(s,a)]  (off-policy)
    - SARSA: Q(s,a) ← Q(s,a) + α[r + γ Q(s',a') - Q(s,a)]  (on-policy)
    
    Objetivo: Implementar el método update() con la regla SARSA.
    """
    
    def update(self, state: int, action: int, reward: float, next_state: int, next_action: int, done: bool):
        """
        TODO: Implementa la actualización SARSA.
        
        Nota: SARSA necesita next_action (la acción que se tomará en s')
        En lugar de max Q(s',a'), usa Q(s', next_action)
        """
        # TODO: Tu código aquí
        pass

def test_ejercicio_2():
    env = gym.make('FrozenLake-v1', map_name="4x4", is_slippery=False)
    sarsa_agent = SARSAAgent(
        n_states=env.observation_space.n,
        n_actions=env.action_space.n,
        learning_rate=0.1,
        discount_factor=0.99
    )
    
    # Entrenar brevemente
    sarsa_agent.train(env, n_episodes=100, verbose=False)
    
    # Verificar que la tabla Q se actualizó
    assert np.sum(np.abs(sarsa_agent.q_table)) > 0, "❌ La tabla Q no se actualizó"
    print("✅ ¡Excelente! SARSA implementado correctamente.")
    print("   SARSA es on-policy y puede ser más estable en ambientes estocásticos.")
    env.close()
    return True

# test_ejercicio_2()  # Descomenta para probar

### 🔴 Ejercicio 3: Análisis de Convergencia

Analiza cómo la tabla Q converge a lo largo del entrenamiento.

In [ ]:
def ejercicio_3_convergence_analysis():
    """
    Objetivo: Estudiar la convergencia de Q-Learning.
    
    Instrucciones:
    1. Modifica QLearningAgent para guardar snapshots de la Q-table cada 100 episodios
    2. Calcula la norma Frobenius de la diferencia entre Q-tables consecutivas
    3. Grafica esta diferencia vs episodios (debería decrecer)
    4. Retorna la figura de plotly
    
    Hint: ||Q_t - Q_{t-1}||_F = sqrt(sum((Q_t - Q_{t-1})^2))
    """
    # TODO: Tu código aquí
    pass

def test_ejercicio_3():
    fig = ejercicio_3_convergence_analysis()
    assert fig is not None, "❌ Debes retornar una figura"
    print("✅ ¡Excelente! Has analizado la convergencia de Q-Learning.")
    print("   La diferencia entre Q-tables sucesivas debería decrecer con el tiempo.")
    return True

# test_ejercicio_3()  # Descomenta para probar

## 📚 Resumen

### Conceptos Clave

- **Q-Learning**: Algoritmo model-free que aprende Q*(s,a) directamente de la experiencia
- **Temporal Difference (TD)**: Aprende de diferencias entre estimaciones sucesivas
- **Error TD**: $\delta = r + \gamma \max_{a'} Q(s',a') - Q(s,a)$
- **Off-Policy**: Q-Learning aprende la política óptima incluso explorando aleatoriamente
- **ε-Greedy**: Balance entre exploración (acción aleatoria) y explotación (mejor acción)
- **Convergencia**: Q-Learning converge a Q* bajo ciertas condiciones

### Algoritmo Q-Learning

```
Inicializar Q(s,a) arbitrariamente
Para cada episodio:
    Inicializar s
    Para cada paso:
        Elegir a usando ε-greedy: a = argmax_a Q(s,a) con prob 1-ε, aleatorio con prob ε
        Tomar a, observar r, s'
        Q(s,a) ← Q(s,a) + α[r + γ max_a' Q(s',a') - Q(s,a)]
        s ← s'
    Hasta s terminal
```

### Ventajas y Desventajas

| Ventaja | Desventaja |
|---------|------------|
| No requiere modelo | Más lento que DP |
| Off-policy (flexible) | Requiere exploración adecuada |
| Simple de implementar | Solo para espacios discretos pequeños |
| Garantías de convergencia | Puede ser inestable con α alto |

### Comparación con Otros Algoritmos

| Algoritmo | Tipo | Actualización | Uso |
|-----------|------|---------------|-----|
| **Value Iteration** | Model-based | $V(s) \leftarrow \max_a \sum_{s'} P(s'|s,a)[R + \gamma V(s')]$ | Modelo conocido |
| **Q-Learning** | Model-free, off-policy | $Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma \max_{a'} Q(s',a') - Q(s,a)]$ | General |
| **SARSA** | Model-free, on-policy | $Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma Q(s',a') - Q(s,a)]$ | Más conservador |

### Lo que viene

Q-Learning funciona bien para problemas pequeños (espacios discretos de estados/acciones), pero:
- ¿Qué pasa si hay millones de estados (imágenes de Atari)?
- ¿Qué pasa si el espacio de estados/acciones es continuo?

En el siguiente notebook veremos **Deep Q-Networks (DQN)**, que usa redes neuronales para aproximar Q(s,a), permitiendo escalar a problemas complejos.

## 🔗 Recursos Adicionales

### 📄 Paper Original

- **"Q-Learning"** - Watkins (1989)
  - PhD thesis que introduce Q-Learning
  - Contexto: Revolucionó RL al no requerir modelo

- **"Q-Learning Convergence"** - Watkins & Dayan (1992)
  - Prueba de convergencia de Q-Learning
  - Machine Learning journal

### 📖 Capítulos de Libros

- **Sutton & Barto - Capítulo 6: Temporal-Difference Learning**
  - Sección 6.5: Q-Learning
  - Explicación detallada con ejemplos

### 🎥 Videos

- **David Silver - Lecture 5: Model-Free Control**
  - https://www.youtube.com/watch?v=0g4j2k_Ggc4
  - Derivación de Q-Learning

### 💻 Implementaciones

- **OpenAI Baselines**
  - https://github.com/openai/baselines
  - Implementaciones de referencia

## ➡️ Próximo Paso

En el siguiente notebook aprenderás sobre **Deep Q-Networks (DQN)**, que combina Q-Learning con deep learning para resolver problemas con espacios de estados grandes o continuos, como juegos de Atari.

**[Continuar con: 04. Deep Q-Networks →](04-deep-q-networks.ipynb)**

---

<div align="center">
    
**¡Has dominado Q-Learning, el corazón del RL clásico! 🎉**

</div>